# CMSC 173 &middot; Module 12 &middot; Example 2
## XOR and the moons

XOR, which no single neuron computes, solved by two hidden neurons chosen by hand. Then module 9's moons, with 1 to 50 hidden neurons, and one network trained epoch by epoch.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides (scikit-learn 1.4.2; a newer version can differ slightly for the trained networks);
change something and see what moves.

**XOR by hand.** $h_1$ is an OR neuron, $h_2$ an AND neuron; the output is "OR but not AND".

In [ ]:
import numpy as np

step = lambda z: (np.asarray(z) >= 0).astype(int)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
h1 = step(X.sum(axis=1) - 0.5)           # OR:  [0 1 1 1]
h2 = step(X.sum(axis=1) - 1.5)           # AND: [0 0 0 1]
print(h1, h2, step(h1 - h2 - 0.5))       # output: [0 1 1 0], XOR

**The moons: more hidden neurons, more bends.** Module 9's RBF SVM scored 0.944 on this split.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=300, noise=0.25, random_state=173)      # module 9's moons
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=173)
nets = {}
for h in [1, 2, 3, 5, 10, 50]:
    nets[h] = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(h,), activation="tanh",
                                                            solver="lbfgs", max_iter=3000, random_state=173))
    print(h, round(nets[h].fit(X_tr, y_tr).score(X_te, y_te), 3))
# 0.889 0.889 0.9 0.933 0.956 0.967 (scikit-learn 1.9 gives 0.956 for 50: training is sensitive)

In [ ]:
import matplotlib.pyplot as plt

gx, gy = np.meshgrid(np.linspace(-1.8, 2.8, 200), np.linspace(-1.3, 1.8, 200))
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, h in zip(axes.ravel(), nets):
    P = nets[h].predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    ax.contourf(gx, gy, P, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, s=10, cmap="coolwarm")
    ax.set_title(f"{h} hidden: test {nets[h].score(X_te, y_te):.3f}")
plt.show()

**One network learning, epoch by epoch.** For about 75 epochs the boundary is nearly straight (test 0.889); then it bends.

In [ ]:
sc = StandardScaler().fit(X_tr)
net = MLPClassifier((10,), activation="tanh", solver="adam", learning_rate_init=0.01, batch_size=32, random_state=173)
for epoch in range(1, 201):
    net.partial_fit(sc.transform(X_tr), y_tr, classes=[0, 1])
    if epoch in (1, 30, 75, 90, 105, 120, 200):
        print(epoch, round(net.loss_, 3), round(net.score(sc.transform(X_te), y_te), 3))
# test 0.889 at epochs 30 and 75, then 0.922, 0.944, 0.956

**Try it.** Give the 50-neuron network a penalty, `alpha=1.0`. Does the boundary stop bending around single points? What happens to the test accuracy?